One problem maybe tokenizing 1000 samples takes about 12 minutes, so the whole dataset will tike 12 * 42 = 504 minutes 

In [10]:
import pandas as pd
import torch
import transformers
from sklearn.model_selection import train_test_split

# Load and rename the column
file_path = '../data/gender.csv'
df = pd.read_csv(file_path)
df.rename(columns={"auhtor_ID": "author_ID"}, inplace=True)

# Check the class distribution (female label)
print(df['female'].value_counts())

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['female'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['female'])

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Check class distribution of splits
print("\nTraining set label distribution:")
print(train_df['female'].value_counts())
print("\nValidation set label distribution:")
print(val_df['female'].value_counts())
print("\nTest set label distribution:")
print(test_df['female'].value_counts())

female
0    23777
1    20858
Name: count, dtype: int64

Training set size: 31244
Validation set size: 6695
Test set size: 6696

Training set label distribution:
female
0    16644
1    14600
Name: count, dtype: int64

Validation set label distribution:
female
0    3566
1    3129
Name: count, dtype: int64

Test set label distribution:
female
0    3567
1    3129
Name: count, dtype: int64


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Convert text data to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_df['post'])
X_val_tfidf = vectorizer.transform(val_df['post'])
X_test_tfidf = vectorizer.transform(test_df['post'])

# Train a logistic regression classifier
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_tfidf, train_df['female'])

# Evaluate the model
val_predictions = classifier.predict(X_val_tfidf)
test_predictions = classifier.predict(X_test_tfidf)

print("\nValidation Results:")
print(classification_report(val_df['female'], val_predictions))

print("\nTest Results:")
print(classification_report(test_df['female'], test_predictions))


Validation Results:
              precision    recall  f1-score   support

           0       0.77      0.82      0.80        79
           1       0.79      0.73      0.76        71

    accuracy                           0.78       150
   macro avg       0.78      0.78      0.78       150
weighted avg       0.78      0.78      0.78       150


Test Results:
              precision    recall  f1-score   support

           0       0.90      0.82      0.86        80
           1       0.82      0.90      0.86        70

    accuracy                           0.86       150
   macro avg       0.86      0.86      0.86       150
weighted avg       0.86      0.86      0.86       150



In [ ]:
from transformers import DistilBertModel, DistilBertTokenizer
from transformers import DistilBertTokenizerFast

# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Freeze...
for param in model.parameters():
    param.requires_grad = False

# Function to generate embeddings using DistilBERT
def generate_embeddings(texts, batch_size=32):
    all_embeddings = []

    # Process texts in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings.cpu())  # Move embeddings back to CPU for storage

    return torch.cat(all_embeddings, dim=0)

# For now just cut off the posts text after 512 tokens
train_embeddings = generate_embeddings(train_df['post'].tolist())

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Convert embeddings and labels to numpy arrays
X_train = train_embeddings.numpy()
y_train = train_df['female'].values

# Create and train the logistic regression model
classifier = LogisticRegression(max_iter=100)
classifier.fit(X_train, y_train)

# Evaluate the model
X_val = generate_embeddings(val_df['post'].tolist()).numpy()
y_val = val_df['female'].values
y_pred = classifier.predict(X_val)

# Calculate accuracy
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.81      0.78        79
           1       0.77      0.69      0.73        71

    accuracy                           0.75       150
   macro avg       0.75      0.75      0.75       150
weighted avg       0.75      0.75      0.75       150

